# LIBRARY

In [ ]:
!pip install vaderSentiment

In [ ]:
import pandas as pd
import re
import emoji
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# LABELING DATA

Memuat Data

In [ ]:
# load dataset
df = pd.read_csv('drive/MyDrive/My room/skripsi/chatgpt1.csv')
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50001 entries, 0 to 50000
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Datetime        50001 non-null  object
 1   Tweet Id        50001 non-null  int64 
 2   Text            50001 non-null  object
 3   Username        50001 non-null  object
 4   Permalink       50001 non-null  object
 5   User            50001 non-null  object
 6   Outlinks        19942 non-null  object
 7   CountLinks      19942 non-null  object
 8   ReplyCount      50001 non-null  int64 
 9   RetweetCount    50001 non-null  int64 
 10  LikeCount       50001 non-null  int64 
 11  QuoteCount      50001 non-null  int64 
 12  ConversationId  50001 non-null  int64 
 13  Language        50001 non-null  object
 14  Source          50001 non-null  object
 15  Media           9502 non-null   object
 16  QuotedTweet     3563 non-null   object
 17  MentionedUsers  17169 non-null  object
 18  hashta

,Datetime,Tweet Id,Text,Username,Permalink,User,Outlinks,CountLinks,ReplyCount,RetweetCount,LikeCount,QuoteCount,ConversationId,Language,Source,Media,QuotedTweet,MentionedUsers,hashtag,hastag_counts
0,2023-01-22 13:44:34+00:00,1617156270871699456,ChatGPTで遊ぶの忘れてた！！\n書類作るコード書いてみてほしいのと、\nどこまで思考整...,mochico0123,https://twitter.com/mochico0123/status/1617156...,https://twitter.com/mochico0123,NaN,NaN,1,0,5,0,1617156270871699456,ja,"<a href=""http://twitter.com/download/iphone"" r...",NaN,NaN,NaN,[],0
1,2023-01-22 13:44:39+00:00,1617156291046133761,@AlexandrovnaIng Prohibition of ChatGPT has be...,Caput_LupinumSG,https://twitter.com/Caput_LupinumSG/status/161...,https://twitter.com/Caput_LupinumSG,NaN,NaN,1,0,5,0,1617148639993806848,en,"<a href=""http://twitter.com/download/iphone"" r...",NaN,NaN,"[User(username='AlexandrovnaIng', id=282705900...",[],0
2,2023-01-22 13:44:44+00:00,1617156308926349312,"Schaut Euch an, was @fobizz @DianaKnodel alles...",ciffi,https://twitter.com/ciffi/status/1617156308926...,https://twitter.com/ciffi,['https://us02web.zoom.us/webinar/register/801...,"['https://t.co/DsoeVJrPBp', 'https://t.co/HflT...",0,0,4,0,1617156308926349312,de,"<a href=""http://twitter.com/#!/download/ipad"" ...",[Photo(previewUrl='https://pbs.twimg.com/media...,https://twitter.com/DianaKnodel/status/1617153...,"[User(username='fobizz', id=884708145792253952...",['#ChatGPT'],1
3,2023-01-22 13:44:49+00:00,1617156332297256961,Bow down to chatGPT 🫡..... https://t.co/ENTSzi...,Vishwasrisiri,https://twitter.com/Vishwasrisiri/status/16171...,https://twitter.com/Vishwasrisiri,['https://twitter.com/agadmator/status/1617155...,['https://t.co/ENTSzi2AQ9'],0,0,2,0,1617156332297256961,en,"<a href=""http://twitter.com/download/android"" ...",NaN,https://twitter.com/agadmator/status/161715501...,NaN,[],0
4,2023-01-22 13:44:52+00:00,1617156345064570880,"Profilinde vatan, Türkiye falan yazan bireyler...",0xGenetikciniz,https://twitter.com/0xGenetikciniz/status/1617...,https://twitter.com/0xGenetikciniz,NaN,NaN,0,0,4,0,1617156345064570880,tr,"<a href=""http://twitter.com/download/iphone"" r...",NaN,NaN,NaN,[],0


Data Filtering/Cleansing

In [ ]:
print("Sebelum filter:", len(df))

df = df[df['Language'] == 'en'].copy()

print("Sesudah filter:", len(df))

Sebelum filter: 50001
Sesudah filter: 32076


Membersihkan text - Menghapus URLs, @mentions, #hashtags

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # Remove #hashtags (keep the word, remove #)
    text = re.sub(r'#(\w+)', r'\1', text)
    # Remove emoji
    text = emoji.replace_emoji(text, replace="")
    # Remove characters other than letters, numbers, and spaces
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Cleaned_Text'] = df['Text'].apply(clean_text)
df[['Text','Cleaned_Text']].head()

,Text,Cleaned_Text
1,@AlexandrovnaIng Prohibition of ChatGPT has be...,Prohibition of ChatGPT has been added to the h...
3,Bow down to chatGPT 🫡..... https://t.co/ENTSzi...,Bow down to chatGPT
6,ChatGPT runs 10K Nvidia training GPUs with pot...,ChatGPT runs 10K Nvidia training GPUs with pot...
7,@SWENGDAD There is repetitive work in every jo...,There is repetitive work in every job there ar...
8,I created a fictional jewelry brand using Chat...,I created a fictional jewelry brand using Chat...


Basic Tokenization

In [ ]:
#Basic Tokenization
df["Tokens"] = df["Cleaned_Text"].apply(lambda x: x.split())
#word count
df["Word_Count"] = df["Tokens"].apply(len)
#Text For VADER
df["Text_For_VADER"] = df["Tokens"].apply(lambda x: " ".join(x))

In [ ]:

display(df[['Cleaned_Text', 'Tokens', 'Word_Count','Text_For_VADER']].head())


,Cleaned_Text,Tokens,Word_Count,Text_For_VADER
1,Prohibition of ChatGPT has been added to the h...,"[Prohibition, of, ChatGPT, has, been, added, t...",15,Prohibition of ChatGPT has been added to the h...
3,Bow down to chatGPT,"[Bow, down, to, chatGPT]",4,Bow down to chatGPT
6,ChatGPT runs 10K Nvidia training GPUs with pot...,"[ChatGPT, runs, 10K, Nvidia, training, GPUs, w...",11,ChatGPT runs 10K Nvidia training GPUs with pot...
7,There is repetitive work in every job there ar...,"[There, is, repetitive, work, in, every, job, ...",29,There is repetitive work in every job there ar...
8,I created a fictional jewelry brand using Chat...,"[I, created, a, fictional, jewelry, brand, usi...",12,I created a fictional jewelry brand using Chat...


Inisialisasi Vader

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    scores = analyzer.polarity_scores(text)
    return scores["compound"]

df["Compound_Score"] = df["Text_For_VADER"].apply(get_sentiment)

Fungsi Labeling

In [ ]:
def label_sentiment(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['Sentiment'] = df['Compound_Score'].apply(label_sentiment)


Hasil Labeling

In [ ]:
print("\n" + "="*50)
print("Sentiment Value Counts:")
print("="*50)
print(df['Sentiment'].value_counts())
print("\n" + "="*50)
print("First 5 rows (Original Text, Cleaned Text, Sentiment):")
print("="*50)
print(df[['Text', 'Cleaned_Text', 'Sentiment']].head())


Sentiment Value Counts:
Sentiment
Positive    15497
Neutral     10877
Negative     5702
Name: count, dtype: int64

First 5 rows (Original Text, Cleaned Text, Sentiment):
                                                Text  \
1  @AlexandrovnaIng Prohibition of ChatGPT has be...   
3  Bow down to chatGPT 🫡..... https://t.co/ENTSzi...   
6  ChatGPT runs 10K Nvidia training GPUs with pot...   
7  @SWENGDAD There is repetitive work in every jo...   
8  I created a fictional jewelry brand using Chat...   

                                        Cleaned_Text Sentiment  
1  Prohibition of ChatGPT has been added to the h...  Positive  
3                                Bow down to chatGPT   Neutral  
6  ChatGPT runs 10K Nvidia training GPUs with pot...   Neutral  
7  There is repetitive work in every job there ar...  Negative  
8  I created a fictional jewelry brand using Chat...  Positive  


Menyimpan Data Labeling ke Bentuk CSV

In [ ]:
df.to_csv('drive/MyDrive/My room/skripsi/data_labeling_6.csv', index=False)


In [ ]:
print(f"\nTotal tweets processed: {len(df)}")
print(f"Languages in dataset: {df['Language'].nunique()} unique languages")
print(f"Language distribution:\n{df['Language'].value_counts().head(10)}")


Total tweets processed: 32076
Languages in dataset: 1 unique languages
Language distribution:
Language
en    32076
Name: count, dtype: int64
